In [1]:
import pandas as pd
import requests
import time

# Load the clean mutations we saved in Phase 2
df = pd.read_csv("results/fh_mutations_clean.csv")
print(f"Loaded {len(df)} variants")
print(df[["gene","cdna_change","chrom","pos"]].head(5))

# Test Ensembl REST API connection
test_url = "https://rest.ensembl.org/sequence/region/human/19:11113547..11113847"
headers = {"Content-Type": "text/plain"}
resp = requests.get(test_url, headers=headers)

if resp.status_code == 200:
    print(f"\nEnsembl connection: OK")
    print(f"Sample sequence (first 60bp): {resp.text[:60]}")
else:
    print(f"Connection failed: {resp.status_code}")

Loaded 27 variants
   gene cdna_change  chrom       pos
0  LDLR    c.408C>G     19  11105314
1  LDLR    c.155G>T     19  11100310
2  LDLR    c.457T>C     19  11105363
3  LDLR   c.1840T>G     19  11116993
4  LDLR   c.1054T>G     19  11110765

Ensembl connection: OK
Sample sequence (first 60bp): CAGAGCCCACGGCGTCTCTTCCTATGACACCGTCATCAGCAGAGACATCCAGGCCCCCGA


In [2]:
def get_sequence(chrom, pos, window=300):
    """
    Fetch a DNA sequence window around a genomic position from Ensembl.
    Returns the sequence and the index of the variant within it.
    """
    start = int(pos) - window
    end   = int(pos) + window
    
    url = f"https://rest.ensembl.org/sequence/region/human/{chrom}:{start}..{end}"
    headers = {"Content-Type": "text/plain"}
    
    resp = requests.get(url, headers=headers)
    
    if resp.status_code == 200:
        return resp.text.strip(), window  # window = position of variant in sequence
    else:
        return None, None


def introduce_mutation(wt_seq, position, ref, alt):
    """
    Introduce a point mutation into a wild-type sequence.
    position: index of the variant within the sequence (0-based)
    """
    if not wt_seq:
        return None
    
    seq = list(wt_seq)
    
    # Verify the reference base matches what we expect
    actual = seq[position].upper()
    
    if ref and actual != ref.upper():
        # Try the complement (some variants are on reverse strand)
        complements = {"A":"T","T":"A","G":"C","C":"G"}
        comp = complements.get(ref.upper(), "?")
        if actual == comp:
            # Reverse strand — flip the alt too
            alt = complements.get(alt.upper(), alt)
        # Continue anyway — coordinates may still be usable
    
    # Introduce the mutation
    if alt:
        seq[position] = alt.upper()
    
    return "".join(seq)


# Test on the first variant
row = df.iloc[0]
print(f"Testing on: {row['gene']} {row['cdna_change']}")
print(f"Chromosome: {row['chrom']}, Position: {row['pos']}")

wt_seq, var_pos = get_sequence(row["chrom"], row["pos"], window=300)

if wt_seq:
    print(f"\nWT sequence retrieved: {len(wt_seq)} bp")
    print(f"Variant position in window: {var_pos}")
    print(f"Base at variant position: {wt_seq[var_pos]}")
    print(f"WT window (middle 30bp): ...{wt_seq[285:315]}...")
else:
    print("Failed to retrieve sequence")

Testing on: LDLR c.408C>G
Chromosome: 19, Position: 11105314

WT sequence retrieved: 601 bp
Variant position in window: 300
Base at variant position: C
WT window (middle 30bp): ...CCGGGACTGCTTGGACGGCTCAGACGAGGC...


In [3]:
results = []

for i, row in df.iterrows():
    print(f"Fetching {i+1}/27: {row['gene']} {row['cdna_change']}...")
    
    # Fetch wild-type sequence
    wt_seq, var_pos = get_sequence(row["chrom"], row["pos"], window=300)
    
    if wt_seq is None:
        print(f"  FAILED — skipping")
        continue
    
    # Parse ref and alt from cdna_change (e.g. c.408C>G → ref=C, alt=G)
    cdna = row["cdna_change"]
    ref, alt = "", ""
    if ">" in cdna:
        change_part = cdna.split(".")[-1]  # e.g. "408C>G"
        try:
            # Extract letters around the >
            ref = "".join([c for c in change_part.split(">")[0] if c.isalpha()])
            alt = "".join([c for c in change_part.split(">")[1] if c.isalpha()])
        except:
            pass
    
    # Introduce the mutation
    mut_seq = introduce_mutation(wt_seq, var_pos, ref, alt)
    
    # Check if mutation actually changed anything
    changed = (wt_seq != mut_seq) if mut_seq else False
    
    results.append({
        "clinvar_id":   row["clinvar_id"],
        "gene":         row["gene"],
        "cdna_change":  row["cdna_change"],
        "protein_change": row["protein_change"],
        "chrom":        row["chrom"],
        "pos":          row["pos"],
        "ref":          ref,
        "alt":          alt,
        "wt_seq":       wt_seq,
        "mut_seq":      mut_seq,
        "var_pos":      var_pos,
        "seq_changed":  changed,
    })
    
    # Be polite to Ensembl API — small pause between requests
    time.sleep(0.3)

df_seqs = pd.DataFrame(results)

print(f"\nSequences retrieved: {len(df_seqs)}")
print(f"Mutations introduced successfully: {df_seqs['seq_changed'].sum()}")
print(f"Failed/unchanged: {(~df_seqs['seq_changed']).sum()}")
print(f"\nSample:")
print(df_seqs[["gene","cdna_change","ref","alt","seq_changed"]].head(10))

Fetching 1/27: LDLR c.408C>G...
Fetching 2/27: LDLR c.155G>T...
Fetching 3/27: LDLR c.457T>C...
Fetching 4/27: LDLR c.1840T>G...
Fetching 5/27: LDLR c.1054T>G...
Fetching 6/27: LDLR c.1961T>G...
Fetching 7/27: LDLR c.1865A>T...
Fetching 8/27: LDLR c.1724T>A...
Fetching 9/27: LDLR c.1074T>G...
Fetching 10/27: LDLR c.986G>C...
Fetching 11/27: LDLR c.648T>G...
Fetching 12/27: LDLR c.398A>G...
Fetching 13/27: LDLR c.483C>G...
Fetching 14/27: LDLR c.1067A>G...
Fetching 15/27: LDLR NM_000527.5(LDLR):c.818-1G>T...
Fetching 16/27: LDLR c.1297G>A...
Fetching 17/27: LDLR c.325T>A...
Fetching 18/27: LDLR c.348C>A...
Fetching 19/27: LDLR c.632A>G...
Fetching 20/27: LDLR c.270T>G...
Fetching 21/27: LDLR NM_000527.5(LDLR):c.68-2A>C...
Fetching 22/27: APOB c.9721G>T...
Fetching 23/27: APOB c.11470C>T...
Fetching 24/27: APOB c.6022G>T...
Fetching 25/27: APOB c.5303C>A...
Fetching 26/27: APOB c.8602G>T...
Fetching 27/27: PCSK9 NM_174936.4(PCSK9):c.399+1G>A...

Sequences retrieved: 27
Mutations introduc

In [4]:
# Save sequences to CSV
# Note: sequences are long so we save a summary version and a full version
df_seqs.to_csv("results/fh_sequences.csv", index=False)

# Save a readable summary without the long sequences
df_summary = df_seqs.drop(columns=["wt_seq", "mut_seq"])
df_summary.to_csv("results/fh_sequences_summary.csv", index=False)

print("Files saved:")
print("  results/fh_sequences.csv         — full data including sequences")
print("  results/fh_sequences_summary.csv — summary without sequences")
print()

# Quick sanity check — show one WT vs mutant comparison
sample = df_seqs.iloc[0]
pos = sample["var_pos"]
print(f"Sanity check — {sample['gene']} {sample['cdna_change']}:")
print(f"  WT  at position {pos}: ...{sample['wt_seq'][pos-5:pos+6]}...")
print(f"  MUT at position {pos}: ...{sample['mut_seq'][pos-5:pos+6]}...")
print(f"  Change: {sample['ref']} → {sample['alt']}")
print()
print("Phase 3 complete!")

Files saved:
  results/fh_sequences.csv         — full data including sequences
  results/fh_sequences_summary.csv — summary without sequences

Sanity check — LDLR c.408C>G:
  WT  at position 300: ...TTGGACGGCTC...
  MUT at position 300: ...TTGGAGGGCTC...
  Change: C → G

Phase 3 complete!
